# B2-022 — Practice p17

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** core · **Time budget:** 65 minutes  
**Concepts:** autoencoder

## Task

Train a tiny MLP autoencoder on the seeded `mixture8d` data and certify held-out reconstruction improvement.

**Data.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `latent_data.train_tensor("mixture8d")` of shape `(48,8)` and `latent_data.heldout_tensor("mixture8d")` of shape `(12,8)`, built from the immutable `latent_data.TRAIN_IDS["mixture8d"]` and `latent_data.HELDOUT_IDS["mixture8d"]` in stored order. Pass training rows to the model exactly as returned (same float32 values; no normalization, noise, shuffling, or concatenation), so every row the model's `forward` receives during training hashes, with `latent_data.row_sha256`, into the training part of `latent_data.ROW_SHA256["mixture8d"]`. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice.

**Model.** `class TinyAutoencoder(nn.Module)` creates, in this order, `self.encoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))` and `self.decoder = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 8))`, with methods `encode(x)`, `decode(h)`, and `forward(x)` returning `decode(encode(x))` of shape `(B,8)`. Call `torch.manual_seed(SEED)` immediately before `model = TinyAutoencoder()`.

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_autoencoder(model, batch, optimizer)` that performs **exactly 400** updates on `batch` with loss `F.mse_loss(model(batch), batch)` (default mean reduction over all entries) and returns a list of 400 Python floats, the loss of each update before its step. Each update runs, in this order: `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()`. Full batch, no shuffling, no scheduler, no gradient clipping, no accumulation, no early stopping. Call it as `trace = train_autoencoder(model, train_rows(), optimizer)`.

**Parts.**
(A) Before training, record `heldout_before = F.mse_loss(model(H), H)` under `torch.no_grad()` with `H = heldout_rows()`, and a deep copy of every parameter.
(B) Train, then record `heldout_after` the same way.
(C) Certify: `len(trace) == 400`; `trace[-1] < trace[0]`; `heldout_after <= 0.10 * heldout_before`; `heldout_after <= 0.05`; at least one parameter tensor changed; every row of `train_rows()` hashes into the training part of `latent_data.ROW_SHA256["mixture8d"]` and none into the held-out part.
(D) In three sentences, explain why the held-out comparison, not the training trace, is the generalization certificate, and why the `0.05` absolute bound is meaningful relative to the "predict the training mean" baseline (compute that baseline's held-out MSE).

**Required answer-check assertions.** Every item of (C), the shapes `(48,8)` and `(12,8)`, float32 dtypes, and the encoder output shape `(12,2)` on held-out rows.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.